# Script for Hackathon

Within each cycle of active learning, you can:

1. Collect training data (original training data + your query data).

2. Train a prediction model to predict the DMS_score for each mutant (e.g., M0A).

3. Use the trained model to predict the score for all mutant in the test set.

4. Select query mutants for next round based on certain criteria. You may want to make sure you don't query the same mutant twice as you only have a limited chances of making queries in total.

In [ ]:
# Install ESM (fair-esm) if not already installed
!pip install fair-esm -q

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 93.1/93.1 kB 9.4 MB/s eta 0:00:00


In [ ]:

import torch
import torch.nn as nn
import torch.optim as optim
import numpy as np
from torch.utils.data import DataLoader, Dataset
import random
from copy import deepcopy
import pandas as pd
from scipy.stats import spearmanr
import argparse
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LinearRegression
#add esm
import esm

In [ ]:
# Set random seeds for reproducibility
SEED = 42
torch.manual_seed(SEED)
np.random.seed(SEED)
random.seed(SEED)

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Using device: {device}')

Using device: cuda


## 1. collect training data

Upload `sequence.fasta`, `train.csv`, `test.csv`, `query1_results.csv`, `query2_results.csv`, and `query3_results.csv` to the current runtime:

1. click the folder icon on the left

2. click the upload icon and upload the files to the current directory

In [ ]:
with open('sequence.fasta', 'r') as f:
  data = f.readlines()

sequence_wt = data[1].strip()
sequence_wt

'MVNEARGNSSLNPCLEGSASSGSESSKDSSRCSTPGLDPERHERLREKMRRRLESGDKWFSLEFFPPRTAEGAVNLISRFDRMAAGGPLYIDVTWHPAGDPGSDKETSSMMIASTAVNYCGLETILHMTCCRQRLEEITGHLHKAKQLGLKNIMALRGDPIGDQWEEEEGGFNYAVDLVKHIRSEFGDYFDICVAGYPKGHPEAGSFEADLKHLKEKVSAGADFIITQLFFEADTFFRFVKACTDMGITCPIVPGIFPIQGYHSLRQLVKLSKLEVPQEIKDVIEPIKDNDAAIRNYGIELAVSLCQELLASGLVPGLHFYTLNREMATTEVLKRLGMWTEDPRRPLPWALSAHPKRREEDVRPIFWASRPKSYIYRTQEWDEFPNGRWGNSSSPAFGELKDYYLFYLKSKSPKEELLKMWGEELTSEESVFEVFVLYLSGEPNRNGHKVTCLPWNDEPLAAETSLLKEELLRVNRQGILTINSQPNINGKPSSDPIVGWGPSGGYVFQKAYLEFFTSRETAEALLQVLKKYELRVNYHLVNVKGENITNAPELQPNAVTWGIFPGREIIQPTVVDPVSFMFWKDEAFALWIERWGKLYEEESPSRTIIQYIHDNYFLVNLVDNDFPLDNCLWQVVEDTLELLNRPTQNARETEAP'

In [ ]:
len(sequence_wt)

656

In [ ]:
def get_mutated_sequence(mut, sequence_wt):
  #this parses mutant string from mut and applies substitution to sequence
  #mutant string like 'M0A' where (wt_aa, position, mut_aa)
  #this is og but other is diff
  wt, pos, mt = mut[0], int(mut[1:-1]), mut[-1]

  sequence = deepcopy(sequence_wt)

  return sequence[:pos]+mt+sequence[pos+1:]

In [ ]:
df_train = pd.read_csv('train.csv')
df_train['sequence'] = df_train.mutant.apply(lambda x: get_mutated_sequence(x, sequence_wt))
print(f'Training samples: {len(df_train)}')
df_train.head()

Training samples: 1140


,mutant,DMS_score,sequence
0,M0Y,0.2730,YVNEARGNSSLNPCLEGSASSGSESSKDSSRCSTPGLDPERHERLR...
1,M0W,0.2857,WVNEARGNSSLNPCLEGSASSGSESSKDSSRCSTPGLDPERHERLR...
2,M0V,0.2153,VVNEARGNSSLNPCLEGSASSGSESSKDSSRCSTPGLDPERHERLR...
3,M0T,0.3122,TVNEARGNSSLNPCLEGSASSGSESSKDSSRCSTPGLDPERHERLR...
4,M0S,0.2180,SVNEARGNSSLNPCLEGSASSGSESSKDSSRCSTPGLDPERHERLR...


In [ ]:
df_test = pd.read_csv('test.csv')
df_test['sequence'] = df_test.mutant.apply(lambda x: get_mutated_sequence(x, sequence_wt))
print(f'Test samples: {len(df_test)}')
df_test.head()

Test samples: 11324


,mutant,sequence
0,V1D,MDNEARGNSSLNPCLEGSASSGSESSKDSSRCSTPGLDPERHERLR...
1,V1Y,MYNEARGNSSLNPCLEGSASSGSESSKDSSRCSTPGLDPERHERLR...
2,V1C,MCNEARGNSSLNPCLEGSASSGSESSKDSSRCSTPGLDPERHERLR...
3,V1A,MANEARGNSSLNPCLEGSASSGSESSKDSSRCSTPGLDPERHERLR...
4,V1E,MENEARGNSSLNPCLEGSASSGSESSKDSSRCSTPGLDPERHERLR...


## Query Data Integration

In [ ]:
# Query 1 Integration
query1_results_path = 'query1_results.csv'

try:
    df_query1 = pd.read_csv(query1_results_path)

    df_query1['sequence'] = df_query1.mutant.apply(lambda x: get_mutated_sequence(x, sequence_wt))
    df_train = pd.concat([df_train, df_query1], ignore_index=True)

    print(f"Successfully integrated {len(df_query1)} new samples.")
    print(f"Total training samples: {len(df_train)}")
except FileNotFoundError:
    print(f"Warning: {query1_results_path} not found. Proceeding with initial training data.")

Successfully integrated 100 new samples.
Total training samples: 1240


In [ ]:
# Query 2 Integration

# uncomment below to run once query2_results.csv is uploaded

query2_results_path = 'query2_results.csv'

try:
    df_query2 = pd.read_csv(query2_results_path)

    df_query2['sequence'] = df_query2.mutant.apply(lambda x: get_mutated_sequence(x, sequence_wt))
    df_train = pd.concat([df_train, df_query2], ignore_index=True)

    print(f"Successfully integrated {len(df_query2)} new samples.")
    print(f"Total training samples: {len(df_train)}")
except FileNotFoundError:
    print(f"Warning: {query2_results_path} not found. Proceeding with initial training data.")

Successfully integrated 100 new samples.
Total training samples: 1340


In [ ]:
# Query 3 Integration

# uncomment below to run once query3_results.csv is uploaded

query3_results_path = 'query3_results.csv'

try:
    df_query3 = pd.read_csv(query3_results_path)

    df_query3['sequence'] = df_query3.mutant.apply(lambda x: get_mutated_sequence(x, sequence_wt))
    df_train = pd.concat([df_train, df_query3], ignore_index=True)

    print(f"Successfully integrated {len(df_query3)} new samples.")
    print(f"Total training samples: {len(df_train)}")
except FileNotFoundError:
    print(f"Warning: {query3_results_path} not found. Proceeding with initial training data.")

Successfully integrated 100 new samples.
Total training samples: 1440


## 2. Train a prediction model

Here, we provided a linear regression model and used one-hot encoding to encode each variant. You would need to build your own model to achieve better performances.

Hint: you can perform cross-validation on the training set to evaluate your predictor before making predictions on the test set.

## **Encoding** **Definition**

In [ ]:
#Add ESM-2 Embedding
# extract 3-part feature vector per mutant:
#   [mean_pool(mutant_seq)  |  pos_emb(mutant)  |  pos_emb(mutant) - pos_emb(wildtype)]
# This gives the model: global context + local context + the actual change signal
esm_model, alphabet = esm.pretrained.esm2_t33_650M_UR50D() #updated from esm2_t12_35M params
esm_model = esm_model.to(device)
esm_model.eval()
for param in esm_model.parameters():
    param.requires_grad = False

batch_converter = alphabet.get_batch_converter()
#ESM_EMBED_DIM = 960  # 320 * 3: mean_pool + mut_pos + delta #8M model
#ESM_EMBED_DIM = 1440  # 480 * 3 — for the 35M model
ESM_EMBED_DIM = 3840 # 1280 * 3 - for 650M params so things downstream will change
print('ESM-2 (8M) loaded and frozen.')
print(f'Input feature dim: {ESM_EMBED_DIM} (mean_pool + position + delta)')

Downloading: "https://dl.fbaipublicfiles.com/fair-esm/models/esm2_t33_650M_UR50D.pt" to /root/.cache/torch/hub/checkpoints/esm2_t33_650M_UR50D.pt
Downloading: "https://dl.fbaipublicfiles.com/fair-esm/regression/esm2_t33_650M_UR50D-contact-regression.pt" to /root/.cache/torch/hub/checkpoints/esm2_t33_650M_UR50D-contact-regression.pt
ESM-2 (8M) loaded and frozen.
Input feature dim: 3840 (mean_pool + position + delta)


In [ ]:
def get_mutation_position(mut):
  #get 0-indexed mutation position from mutant string
  # input M0A and get 0
    return int(mut[1:-1])

In [ ]:
def get_esm_repr(sequences_with_labels, seq_len):
    """
    Run ESM-2 on a list of (label, sequence) pairs.
    esm2_t12_35M has 12 transformer layers — we extract from the final layer (12).
    Returns token representations of shape (B, seq_len+2, 480).
    """
    batch_labels, batch_strs, batch_tokens = batch_converter(sequences_with_labels)
    batch_tokens = batch_tokens.to(device)
    with torch.no_grad():
        results = esm_model(batch_tokens, repr_layers=[33], return_contacts=False) #changed repr_layer to 33 for 650M params
    return results['representations'][33]  # (B, L+2, 480) #changed repr_layer to 33 for 650M params


In [ ]:
def extract_esm_embeddings(df, sequence_wt, mutant_col='mutant', seq_col='sequence', batch_size=32):
  #extract a 3part esm2 feature vector for ecah mutant
  #1-mean pool -  mean of all residue embeddings of the mutant sequence  (320-dim)
  #2-mutation position - embedding at the mutated position of the mutant seq    (320-dim)
  #3-delta - mut_pos minus the same position in the wildtype seq    (320-dim)
  #then concatenated into 960-dim input per sample
  #delta is what encodes what changed at the biochem level
    seq_len = len(sequence_wt)
    all_embeddings = []

    # Pre-compute wildtype position embeddings for all positions (run WT once per batch)
    wt_data = [('wt', sequence_wt)]
    wt_repr = get_esm_repr(wt_data, seq_len)  # (1, seq_len+2, 320)
    wt_repr = wt_repr[0]  # (seq_len+2, 320)

    for start in range(0, len(df), batch_size):
        batch_df = df.iloc[start:start + batch_size]
        data_batch = [(row[mutant_col], row[seq_col]) for _, row in batch_df.iterrows()]

        mut_repr = get_esm_repr(data_batch, seq_len)  # (B, seq_len+2, 320)

        for i, row in enumerate(batch_df.itertuples()):
            mut_pos = get_mutation_position(row.mutant)  # 0-indexed
            token_idx = mut_pos + 1                      # +1 for <cls> token

            # 1. Mean pool: average over all residue tokens (exclude <cls> at 0 and <eos> at end)
            mean_pool = mut_repr[i, 1:seq_len+1, :].mean(dim=0).cpu().numpy()  # (320,)

            # 2. Position embedding of the mutant at the mutated site
            mut_pos_emb = mut_repr[i, token_idx, :].cpu().numpy()             # (320,)

            # 3. Delta: how much this position changed vs wildtype
            wt_pos_emb = wt_repr[token_idx, :].cpu().numpy()                  # (320,)
            delta = mut_pos_emb - wt_pos_emb                                   # (320,)

            # Concatenate all three parts -> (960,)
            emb = np.concatenate([mean_pool, mut_pos_emb, delta], axis=0)
            all_embeddings.append(emb)

        if (start // batch_size) % 10 == 0:
            print(f'  Processed {min(start + batch_size, len(df))}/{len(df)} sequences...')

    return np.array(all_embeddings, dtype=np.float32)

In [ ]:
#print and extract
print('Extracting training embeddings...')
X_train_emb = extract_esm_embeddings(df_train, sequence_wt)
print(f'Training embeddings shape: {X_train_emb.shape}')  # expect (1340, 3840)

print('Extracting test embeddings...')
X_test_emb = extract_esm_embeddings(df_test, sequence_wt)
print(f'Test embeddings shape: {X_test_emb.shape}')  # expect (11324, 3840)

Extracting training embeddings...
  Processed 32/1440 sequences...
  Processed 352/1440 sequences...
  Processed 672/1440 sequences...
  Processed 992/1440 sequences...
  Processed 1312/1440 sequences...
Training embeddings shape: (1440, 3840)
Extracting test embeddings...
  Processed 32/11324 sequences...
  Processed 352/11324 sequences...
  Processed 672/11324 sequences...
  Processed 992/11324 sequences...
  Processed 1312/11324 sequences...
  Processed 1632/11324 sequences...
  Processed 1952/11324 sequences...
  Processed 2272/11324 sequences...
  Processed 2592/11324 sequences...
  Processed 2912/11324 sequences...
  Processed 3232/11324 sequences...
  Processed 3552/11324 sequences...
  Processed 3872/11324 sequences...
  Processed 4192/11324 sequences...
  Processed 4512/11324 sequences...
  Processed 4832/11324 sequences...
  Processed 5152/11324 sequences...
  Processed 5472/11324 sequences...
  Processed 5792/11324 sequences...
  Processed 6112/11324 sequences...
  Processed

In [ ]:
def get_esm_loglikelihood(df, sequence_wt, mutant_col='mutant'):
    """
    Compute ESM-2 log-likelihood ratio score for each mutant.
    Score = log P(mutant_aa | context) - log P(wt_aa | context)
    This is a zero-shot fitness signal that generalises to all positions
    because it relies only on ESM's pre-trained knowledge, not the training labels.
    Higher score = ESM thinks the mutation is more tolerated.
    """
    seq_len = len(sequence_wt)
    wt_data = [('wt', sequence_wt)]
    _, _, wt_tokens = batch_converter(wt_data)
    wt_tokens = wt_tokens.to(device)

    with torch.no_grad():
        wt_logits = esm_model(wt_tokens, repr_layers=[], return_contacts=False)['logits']
    wt_log_probs = torch.log_softmax(wt_logits[0], dim=-1)  # (seq_len+2, vocab)

    scores = []
    for mut in df[mutant_col]:
        wt_aa  = mut[0]
        pos    = int(mut[1:-1])   # 0-indexed
        mut_aa = mut[-1]
        token_idx = pos + 1       # +1 for <cls>

        mut_idx = alphabet.get_idx(mut_aa)
        wt_idx  = alphabet.get_idx(wt_aa)

        # Log-likelihood ratio: how much more/less likely is the mutant vs wildtype
        score = (wt_log_probs[token_idx, mut_idx] - wt_log_probs[token_idx, wt_idx]).item()
        scores.append(score)

    return np.array(scores, dtype=np.float32)


print('Computing ESM log-likelihood scores for train set...')
ll_train = get_esm_loglikelihood(df_train, sequence_wt)
print('Computing ESM log-likelihood scores for test set...')
ll_test  = get_esm_loglikelihood(df_test, sequence_wt)
print(f'LL scores shape — train: {ll_train.shape}, test: {ll_test.shape}')


Computing ESM log-likelihood scores for train set...
Computing ESM log-likelihood scores for test set...
LL scores shape — train: (1440,), test: (11324,)


# **Dataset &** **DataLoader**

In [ ]:
'''hyperparameters'''
VAL_RATIO   = 0.2    # means 20% of training data used for validation
BATCH_SIZE  = 64
NUM_EPOCHS  = 100
LR          = 3e-4
WEIGHT_DECAY = 1e-2
PATIENCE    = 10


#seq_length = 656
#seed = 0 # seed for splitting the validation set
#val_ratio = 0.2 # proportion of validation set

In [ ]:
class EmbeddingDataset(Dataset):
    """Dataset for pre-computed ESM embeddings."""

    def __init__(self, embeddings, targets=None):
        self.embeddings = torch.tensor(embeddings, dtype=torch.float32)
        self.targets = torch.tensor(targets, dtype=torch.float32) if targets is not None else None

    def __len__(self):
        return len(self.embeddings)

    def __getitem__(self, idx):
        if self.targets is not None:
            return self.embeddings[idx], self.targets[idx]
        return self.embeddings[idx], torch.tensor(0.0)  # dummy target for test set


In [ ]:
# Position-held-out validation split
# Hold out entire positions (not random samples) so val mimics the test distribution
df_train['position'] = df_train['mutant'].apply(lambda x: int(x[1:-1]))
all_positions = df_train['position'].unique()

np.random.seed(SEED)
val_positions = set(np.random.choice(all_positions,
                    size=int(len(all_positions) * 0.2), replace=False))

mask_val = df_train['position'].isin(val_positions)
val_idx   = df_train[mask_val].index.tolist()
train_idx = df_train[~mask_val].index.tolist()

# Build embedding + LL feature arrays for each split
# Concatenate ESM embedding (3840-dim) with LL score (1-dim) -> 3841-dim total
X_all = np.concatenate([X_train_emb, ll_train.reshape(-1, 1)], axis=1)
X_test_full = np.concatenate([X_test_emb, ll_test.reshape(-1, 1)], axis=1)

X_tr  = X_all[train_idx]
X_val = X_all[val_idx]
y_tr  = df_train.loc[train_idx, 'DMS_score'].values.astype(np.float32)
y_val = df_train.loc[val_idx,   'DMS_score'].values.astype(np.float32)

print(f'Train positions: {len(all_positions) - len(val_positions)} | samples: {len(X_tr)}')
print(f'Val positions (unseen): {len(val_positions)} | samples: {len(X_val)}')
print(f'Feature dim: {X_tr.shape[1]} (3840 ESM + 1 log-likelihood)')


Train positions: 174 | samples: 1225
Val positions (unseen): 43 | samples: 215
Feature dim: 3841 (1440 ESM + 1 log-likelihood)


In [ ]:
# Insert between Cell 22 and Cell 23
from sklearn.preprocessing import StandardScaler

scaler = StandardScaler()
X_tr          = scaler.fit_transform(X_tr)        # fit only on training data
X_val         = scaler.transform(X_val)           # apply same scale to val
X_test_full   = scaler.transform(X_test_full)     # and to test
print(f'Features scaled. Mean≈0, Std≈1 across all dims.')

Features scaled. Mean≈0, Std≈1 across all dims.


In [ ]:
train_dataset = EmbeddingDataset(X_tr,        y_tr)
val_dataset   = EmbeddingDataset(X_val,        y_val)
test_dataset  = EmbeddingDataset(X_test_full)  # no labels

train_loader = DataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=True)
val_loader   = DataLoader(val_dataset,   batch_size=BATCH_SIZE, shuffle=False)
test_loader  = DataLoader(test_dataset,  batch_size=BATCH_SIZE, shuffle=False)

print(f'Train: {len(train_dataset)} | Val: {len(val_dataset)} | Test: {len(test_dataset)}')


Train: 1225 | Val: 215 | Test: 11324


# **MLP Model Definition**

In [ ]:
class MLP(nn.Module):
    """
    Two-hidden-layer MLP for DMS score regression.
    Input: 3841-dim = 3840 ESM-2 35M features + 1 ESM log-likelihood score
    Architecture: 3841 -> Linear(512) -> GELU -> Dropout
                       -> Linear(128) -> GELU -> Dropout
                       -> Linear(1)
    """
    def __init__(self, input_dim, hidden_dims=[512, 256, 128], dropout=0.2):
        super().__init__()
        layers = []
        prev_dim = input_dim
        for h in hidden_dims:
            layers += [
                nn.Linear(prev_dim, h),
                nn.BatchNorm1d(h),
                nn.GELU(),           # GELU outperforms ReLU for transformer-derived features
                nn.Dropout(dropout),
            ]
            prev_dim = h
        layers.append(nn.Linear(prev_dim, 1))
        self.network = nn.Sequential(*layers)

    def forward(self, x):
        return self.network(x).squeeze(-1)


FEATURE_DIM = X_tr.shape[1]  # dynamically set from data (3841)
model = MLP(input_dim=FEATURE_DIM).to(device)
print(model)
total_params = sum(p.numel() for p in model.parameters() if p.requires_grad)
print(f'Trainable parameters: {total_params:,}')
print(f'Input feature dim: {FEATURE_DIM}')

MLP(
  (network): Sequential(
    (0): Linear(in_features=3841, out_features=512, bias=True)
    (1): BatchNorm1d(512, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
    (2): GELU(approximate='none')
    (3): Dropout(p=0.2, inplace=False)
    (4): Linear(in_features=512, out_features=256, bias=True)
    (5): BatchNorm1d(256, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
    (6): GELU(approximate='none')
    (7): Dropout(p=0.2, inplace=False)
    (8): Linear(in_features=256, out_features=128, bias=True)
    (9): BatchNorm1d(128, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
    (10): GELU(approximate='none')
    (11): Dropout(p=0.2, inplace=False)
    (12): Linear(in_features=128, out_features=1, bias=True)
  )
)
Trainable parameters: 2,133,249
Input feature dim: 3841


# **Training Definition**

In [ ]:
# Loss, Optimizer, Scheduler
criterion = nn.MSELoss()  # MSE is more sensitive to ranking quality than HuberLoss
optimizer = optim.AdamW(model.parameters(), lr=LR, weight_decay=WEIGHT_DECAY)
scheduler = optim.lr_scheduler.ReduceLROnPlateau(
    optimizer, mode='min', factor=0.5, patience=5)


def train_one_epoch(model, loader, optimizer, criterion):
    model.train()
    total_loss = 0.0
    for X_batch, y_batch in loader:
        X_batch, y_batch = X_batch.to(device), y_batch.to(device)
        optimizer.zero_grad()
        preds = model(X_batch)
        loss = criterion(preds, y_batch)
        loss.backward()
        optimizer.step()
        total_loss += loss.item() * len(X_batch)
    return total_loss / len(loader.dataset)

In [ ]:
#evaluate definition
def evaluate(model, loader, criterion):
    model.eval()
    total_loss = 0.0
    all_preds, all_targets = [], []
    with torch.no_grad():
        for X_batch, y_batch in loader:
            X_batch, y_batch = X_batch.to(device), y_batch.to(device)
            preds = model(X_batch)
            loss = criterion(preds, y_batch)
            total_loss += loss.item() * len(X_batch)
            all_preds.extend(preds.cpu().numpy())
            all_targets.extend(y_batch.cpu().numpy())
    avg_loss = total_loss / len(loader.dataset)
    rho, _ = spearmanr(all_preds, all_targets)
    return avg_loss, rho

# **Train and Evaluate on Validation and Test Sets**

In [ ]:
# Train and evaluate on validation set
N_ENSEMBLE = 10
ensemble_test_preds = []
ensemble_val_preds = []

for seed in range(N_ENSEMBLE):
    torch.manual_seed(seed)
    np.random.seed(seed)
    model = MLP(input_dim=FEATURE_DIM).to(device)
    optimizer = optim.AdamW(model.parameters(), lr=LR, weight_decay=WEIGHT_DECAY)
    scheduler = optim.lr_scheduler.ReduceLROnPlateau(optimizer, mode='max', factor=0.5, patience=5)

    best_val_spearman = -1
    best_state = None
    epochs_no_improve = 0

    for epoch in range(1, NUM_EPOCHS + 1):
        train_loss = train_one_epoch(model, train_loader, optimizer, criterion)
        val_loss, val_spearman = evaluate(model, val_loader, criterion)
        scheduler.step(val_spearman)

        if epoch % 5 == 0 or epoch == 1:
          print(f'Epoch {epoch:3d} | Train Loss: {train_loss:.4f} | Val Loss: {val_loss:.4f} | Val Spearman: {val_spearman:.4f}')

        if val_spearman > best_val_spearman:
            best_val_spearman = val_spearman
            best_state = deepcopy(model.state_dict())
            epochs_no_improve = 0
        else:
            epochs_no_improve += 1
            if epochs_no_improve >= PATIENCE:
                break

    model.load_state_dict(best_state)

    model.eval()
    val_preds, test_preds = [], []
    with torch.no_grad():
        for X_batch, _ in val_loader:
            val_preds.extend(model(X_batch.to(device)).cpu().numpy())
        for X_batch, _ in test_loader:
            test_preds.extend(model(X_batch.to(device)).cpu().numpy())

    ensemble_val_preds.append(val_preds)
    ensemble_test_preds.append(test_preds)
    print(f'Seed {seed} | Best Val Spearman: {best_val_spearman:.4f}')

# ── Average ensemble, then blend with LL using direct scaling (preserves magnitude) ────────
final_val_preds  = np.mean(ensemble_val_preds, axis=0)
final_test_preds = np.mean(ensemble_test_preds, axis=0)

rho, _ = spearmanr(final_val_preds, y_val)
print(f'\nEnsemble Val Spearman: {rho:.4f}')

# Scale both signals to have comparable magnitudes (z-score keeps magnitude differences)
from sklearn.preprocessing import StandardScaler

scaler_mlp = StandardScaler()
scaler_ll = StandardScaler()

# Fit scalers on validation set
mlp_val_scaled = scaler_mlp.fit_transform(final_val_preds.reshape(-1, 1)).ravel()
ll_val_scaled = scaler_ll.fit_transform(ll_train[val_idx].reshape(-1, 1)).ravel()

# Tune alpha blend using scaled (NOT rank-normalized) scores
print("\nTuning blend alpha on validation set (using scaled scores, no rank norm):")
best_alpha, best_rho = 0.75, -1
for alpha in [0.5, 0.55, 0.6, 0.65, 0.7, 0.75, 0.8, 0.85, 0.9, 0.95, 1.0]:
    blended_val = alpha * mlp_val_scaled + (1 - alpha) * ll_val_scaled
    rho, _ = spearmanr(blended_val, y_val)
    print(f'  alpha={alpha:.2f} | Val Spearman: {rho:.4f}')
    if rho > best_rho:
        best_rho, best_alpha = rho, alpha

print(f'\nBest alpha: {best_alpha} | Val Spearman: {best_rho:.4f}')

# Apply best alpha to test predictions (using same scalers fit on validation)
mlp_test_scaled = scaler_mlp.transform(final_test_preds.reshape(-1, 1)).ravel()
ll_test_scaled = scaler_ll.transform(ll_test.reshape(-1, 1)).ravel()
final_test_preds_blended = best_alpha * mlp_test_scaled + (1 - best_alpha) * ll_test_scaled


# Convert to percentile rank [0, 100] for interpretability as fitness score
from scipy.stats import rankdata
percentile_ranks = rankdata(final_test_preds_blended) / len(final_test_preds_blended) * 100
final_test_preds_blended = percentile_ranks
print(f'Final predictions converted to percentile rank [0, 100]')
print(f'  Min: {final_test_preds_blended.min():.1f}, Max: {final_test_preds_blended.max():.1f}, Mean: {final_test_preds_blended.mean():.1f}')

Epoch   1 | Train Loss: 0.2301 | Val Loss: 0.0991 | Val Spearman: 0.6289
Epoch   5 | Train Loss: 0.0810 | Val Loss: 0.0502 | Val Spearman: 0.6846
Epoch  10 | Train Loss: 0.0581 | Val Loss: 0.0423 | Val Spearman: 0.7268
Epoch  15 | Train Loss: 0.0490 | Val Loss: 0.0394 | Val Spearman: 0.7108
Epoch  20 | Train Loss: 0.0365 | Val Loss: 0.0391 | Val Spearman: 0.7106
Epoch  25 | Train Loss: 0.0332 | Val Loss: 0.0385 | Val Spearman: 0.7259
Epoch  30 | Train Loss: 0.0286 | Val Loss: 0.0388 | Val Spearman: 0.7291
Epoch  35 | Train Loss: 0.0286 | Val Loss: 0.0370 | Val Spearman: 0.7301
Epoch  40 | Train Loss: 0.0258 | Val Loss: 0.0399 | Val Spearman: 0.7339
Epoch  45 | Train Loss: 0.0242 | Val Loss: 0.0382 | Val Spearman: 0.7171
Seed 0 | Best Val Spearman: 0.7360
Epoch   1 | Train Loss: 0.3092 | Val Loss: 0.1288 | Val Spearman: 0.6906
Epoch   5 | Train Loss: 0.0800 | Val Loss: 0.0594 | Val Spearman: 0.6956
Epoch  10 | Train Loss: 0.0528 | Val Loss: 0.0476 | Val Spearman: 0.6919
Seed 1 | Best Va

# **Save**

In [ ]:
# Save predictions to CSV

# df_test['DMS_score_predicted'] = final_test_preds_blended

df_test[['mutant', 'DMS_score_predicted']].to_csv('predictions.csv', index=False)
print('Saved predictions.csv.')
# kaggle
# Export with required submission headers: id (1..N), DMS_score
# df_submission = pd.DataFrame({
#     'id': np.arange(0, len(df_test), dtype=int),
#     'DMS_score': df_test['DMS_score_predicted'].values,
# })
# df_submission.to_csv('predictions.csv', index=False)
# print('Saved predictions.csv with headers: id, DMS_score (id is 1..N).')


# Build top10.txt from the 10 highest predicted mutants
top10_df = (
    df_test
    .sort_values('DMS_score_predicted', ascending=False)
    .head(10)
    .copy()
)

# Sanity checks
assert len(top10_df) == 10, f"Expected 10 mutants, got {len(top10_df)}"
assert top10_df['mutant'].isin(df_test['mutant']).all(), "Found a mutant not in test set"

# Write one mutant per line
with open('top10.txt', 'w') as f:
    for mut in top10_df['mutant']:
        f.write(mut + '\n')

print('Saved top10.txt')
print('Top-10 table (highest predicted DMS_score):')
print(top10_df[['mutant', 'DMS_score_predicted']])

Saved predictions.csv.
Saved top10.txt
Top-10 table (highest predicted DMS_score):
      mutant  DMS_score_predicted
10131  R593Q           100.000000
8996   L533N            99.991169
3910   E231N            99.982338
1295    E70A            99.973508
10140  R593D            99.964677
2362   R131N            99.955846
3907   E231S            99.947015
630     P34F            99.938184
9001   L533G            99.929354
10139  R593E            99.920523


In [ ]:
# df_test[['mutant', 'DMS_score_predicted']].to_csv('test_predictions.csv')

## 3. Select query for next round

In [ ]:
# df_test.sort_values('DMS_score_predicted', ascending=False).head(100)

QUERY 1 - DON'T RUN if Query 1 has already been completed (already have query1_results.csv)

In [ ]:
# QUERY_SIZE = 100

# # Collect per-model predictions for uncertainty estimation
# # Shape: (N_ENSEMBLE, n_test_samples)
# preds_array = np.array(ensemble_test_preds)  # (10, n_test)

# # Uncertainty = standard deviation across ensemble members
# # High std = models disagree = we should query this sample
# uncertainty = preds_array.std(axis=0)

# df_test['uncertainty'] = uncertainty
# df_test['position'] = df_test['mutant'].apply(lambda x: int(x[1:-1]))

# print(f"Uncertainty stats — mean: {uncertainty.mean():.4f}, "
#       f"max: {uncertainty.max():.4f}, min: {uncertainty.min():.4f}")


# already_queried = set(df_train['mutant'].values)  # never re-query

# df_candidates = df_test[~df_test['mutant'].isin(already_queried)].copy()
# print(f"Candidates after removing already-seen mutants: {len(df_candidates)}")

# # One mutant per position (the most uncertain one)
# best_per_position = (
#     df_candidates
#     .sort_values('uncertainty', ascending=False)
#     .groupby('position')
#     .first()
#     .reset_index()
#     .sort_values('uncertainty', ascending=False)
# )

# # Take top-QUERY_SIZE diverse candidates.
# # If there aren't enough unique positions, fall back to next-best same-position.

# if len(best_per_position) >= QUERY_SIZE:
#     querys = best_per_position.head(QUERY_SIZE)['mutant'].values
# else:
#     used = set(best_per_position['mutant'].values)
#     remaining = (df_candidates[~df_candidates['mutant'].isin(used)]
#                  .sort_values('uncertainty', ascending=False))
#     extra_needed = QUERY_SIZE - len(best_per_position)
#     extra = remaining.head(extra_needed)['mutant'].values
#     querys = np.concatenate([best_per_position['mutant'].values, extra])

# print(f"\nSelected {len(querys)} mutants to query.")
# print(f"Unique positions covered: {len(set(df_test[df_test['mutant'].isin(querys)]['position']))}")
# print(f"Sample queries: {querys[:5]}")

# with open('query.txt', 'w') as f:
#     for mutant in querys:
#         f.write(mutant + '\n')

Uncertainty stats — mean: 0.0691, max: 0.3425, min: 0.0165
Candidates after removing already-seen mutants: 11224

Selected 100 mutants to query.
Unique positions covered: 100
Sample queries: ['N2R' 'T33C' 'C31K' 'N11C' 'K104F']


QUERY 2 - DON'T RUN if Query 2 has already been completed (already have query2_results.csv)

In [ ]:
# ## QUERY 2 — UCB acquisition strategy

# QUERY_SIZE = 100
# BETA = 1.0  # exploration-exploitation tradeoff; tune between 0.5 and 2.0

# # Ensure ensemble predictions are available
# preds_array = np.array(ensemble_test_preds)  # (N_ENSEMBLE, n_test)
# mu    = preds_array.mean(axis=0)   # predicted mean fitness
# sigma = preds_array.std(axis=0)    # ensemble uncertainty

# # UCB score: reward high predicted fitness AND high uncertainty
# ucb_scores = mu + BETA * sigma

# df_test['ucb_score']   = ucb_scores
# df_test['mu_pred']     = mu
# df_test['uncertainty'] = sigma
# df_test['position']    = df_test['mutant'].apply(lambda x: int(x[1:-1]))

# # Exclude anything already in training (original + query 1)
# already_seen = set(df_train['mutant'].values)
# df_candidates = df_test[~df_test['mutant'].isin(already_seen)].copy()
# print(f"Candidates: {len(df_candidates)} (excluded {len(already_seen)} already-seen)")

# # One mutant per position — the one with highest UCB score
# # This preserves positional diversity while still exploiting the signal
# best_per_position = (
#     df_candidates
#     .sort_values('ucb_score', ascending=False)
#     .groupby('position')
#     .first()
#     .reset_index()
#     .sort_values('ucb_score', ascending=False)
# )

# if len(best_per_position) >= QUERY_SIZE:
#     querys_q2 = best_per_position.head(QUERY_SIZE)['mutant'].values
# else:
#     used = set(best_per_position['mutant'].values)
#     extra = (df_candidates[~df_candidates['mutant'].isin(used)]
#              .sort_values('ucb_score', ascending=False)
#              .head(QUERY_SIZE - len(best_per_position))['mutant'].values)
#     querys_q2 = np.concatenate([best_per_position['mutant'].values, extra])

# print(f"Selected {len(querys_q2)} mutants for Query 2")
# print(f"Unique positions: {len(set(df_test[df_test['mutant'].isin(querys_q2)]['position']))}")

# # Quick sanity check — what does the selected batch look like?
# selected = df_test[df_test['mutant'].isin(querys_q2)]
# print(f"Selected UCB score  — mean: {selected['ucb_score'].mean():.4f}, max: {selected['ucb_score'].max():.4f}")
# print(f"Selected mu (fitness) — mean: {selected['mu_pred'].mean():.4f}")
# print(f"Selected sigma (uncert) — mean: {selected['uncertainty'].mean():.4f}")

# with open('query2.txt', 'w') as f:
#     for mutant in querys_q2:
#         f.write(mutant + '\n')
# print("Saved query2.txt")

Candidates: 11224 (excluded 1240 already-seen)
Selected 100 mutants for Query 2
Unique positions: 100
Selected UCB score  — mean: 0.8175, max: 1.0043
Selected mu (fitness) — mean: 0.7057
Selected sigma (uncert) — mean: 0.1118
Saved query2.txt


QUERY 3

In [ ]:
# TODO - query the top-predicted unmeasured mutations to refine your top-10 list
# QUERY 3: greedy selection with extra safety checks

QUERY_SIZE = 100

df_query_candidates = df_test.copy()
df_query_candidates["query3_score"] = final_test_preds_blended

seen_mutants = set(df_train["mutant"].astype(str).tolist())

df_query_candidates = df_query_candidates[
    ~df_query_candidates["mutant"].isin(seen_mutants)
].copy()

# drop duplicate mutant rows if any exist
df_query_candidates = df_query_candidates.drop_duplicates(subset="mutant")

df_query3 = (
    df_query_candidates
    .sort_values("query3_score", ascending=False)
    .head(QUERY_SIZE)
    .copy()
)

selected_mutants = df_query3["mutant"].tolist()

assert len(selected_mutants) <= QUERY_SIZE, "More than 100 mutants selected"
assert len(selected_mutants) == len(set(selected_mutants)), "Duplicate mutants found"
assert all(m not in seen_mutants for m in selected_mutants), "A seen mutant was included"

with open("query3.txt", "w") as f:
    f.write("\n".join(selected_mutants))

print(f"Saved query3.txt with {len(selected_mutants)} mutants")
print("Top 10 selected mutants:")
print(df_query3[["mutant", "query3_score"]].head(10).to_string(index=False))

Saved query3.txt with 100 mutants
Top 10 selected mutants:
mutant  query3_score
 L436N      0.997240
  E70S      0.997143
 L533G      0.996830
 R593D      0.995898
 L533D      0.995691
 R593T      0.995602
 R444A      0.995028
 L533A      0.994807
 V435Y      0.994777
 V435Q      0.993412


In [ ]:
# Display top 10 queried mutants by true DMS score (from query result files)
query_files = ['query1_results.csv', 'query2_results.csv', 'query3_results.csv']
query_frames = []

for qf in query_files:
    try:
        qdf = pd.read_csv(qf)
        if 'mutant' in qdf.columns and 'DMS_score' in qdf.columns:
            qdf = qdf[['mutant', 'DMS_score']].copy()
            qdf['source_file'] = qf
            query_frames.append(qdf)
        else:
            print(f"Skipping {qf}: missing mutant/DMS_score columns")
    except FileNotFoundError:
        print(f"{qf} not found, skipping")

if query_frames:
    df_queries = pd.concat(query_frames, ignore_index=True)
    # If a mutant appears multiple times across query files, keep latest file occurrence
    df_queries = df_queries.drop_duplicates(subset='mutant', keep='last')

    top10_queried = (
        df_queries.sort_values('DMS_score', ascending=False)
        .head(10)
        .reset_index(drop=True)
    )

    print("Top 10 queried mutants by true DMS_score:")
    print(top10_queried.to_string(index=False))
else:
    print("No query result files found with mutant and DMS_score columns.")

Top 10 queried mutants by true DMS_score:
mutant  DMS_score        source_file
 S502R   0.994169 query1_results.csv
 K104F   0.990183 query2_results.csv
  E70A   0.989502 query1_results.csv
  R50E   0.989335 query1_results.csv
 V239Y   0.988429 query1_results.csv
 H262A   0.986790 query1_results.csv
 R444Y   0.983788 query1_results.csv
 A649F   0.983128 query1_results.csv
 T652F   0.982428 query2_results.csv
 I160V   0.979235 query2_results.csv


In [ ]:
# Check whether specific mutants were queried and have true DMS scores
mutants_to_check = [
    "R593Q", "R131N", "L533N", "E70A", "R593D",
    "R593T", "R593S", "K490A", "L533D", "F581Y"
]

query_files = ["query1_results.csv", "query2_results.csv", "query3_results.csv"]
found_rows = []

for qf in query_files:
    try:
        qdf = pd.read_csv(qf)
        if "mutant" in qdf.columns and "DMS_score" in qdf.columns:
            hit = qdf[qdf["mutant"].isin(mutants_to_check)][["mutant", "DMS_score"]].copy()
            if len(hit) > 0:
                hit["source_file"] = qf
                found_rows.append(hit)
        else:
            print(f"Skipping {qf}: missing mutant/DMS_score columns")
    except FileNotFoundError:
        print(f"{qf} not found, skipping")

if found_rows:
    df_found = pd.concat(found_rows, ignore_index=True)
    # Keep last occurrence if a mutant appears in multiple query files
    df_found = df_found.drop_duplicates(subset="mutant", keep="last")

    # Preserve your input order for easy scanning
    order_map = {m: i for i, m in enumerate(mutants_to_check)}
    df_found["order_idx"] = df_found["mutant"].map(order_map)
    df_found = df_found.sort_values("order_idx").drop(columns=["order_idx"])

    print("Mutants found in query results with true DMS_score:")
    print(df_found.to_string(index=False))
else:
    print("None of the listed mutants were found in query1/2/3 result files.")

found_set = set(df_found["mutant"]) if found_rows else set()
missing = [m for m in mutants_to_check if m not in found_set]
if missing:
    print("\nNot found in query result files:")
    print(", ".join(missing))

Mutants found in query results with true DMS_score:
mutant  DMS_score        source_file
 R593Q   0.958571 query1_results.csv
 R131N   0.879604 query2_results.csv
 L533N   0.947330 query2_results.csv
  E70A   0.989502 query1_results.csv
 R593D   0.922901 query3_results.csv
 R593T   0.679036 query3_results.csv
 R593S   0.848568 query3_results.csv
 K490A   0.969630 query3_results.csv
 L533D   0.829035 query3_results.csv
 F581Y   0.976675 query1_results.csv
